# anythreejs visual QA gallery

Run every cell and eyeball each figure. This is the human check that
automation can't replace: colors, lighting, and text should look right,
and orbit/zoom/pan should feel smooth in every figure. Note that
anythreejs ships a much newer three.js than pythreejs did, so lighting
and color management are *expected* to differ slightly from old
pythreejs output.

Checklist per figure: renders at all - colors match the code - orbiting
is smooth - no console errors (open browser dev tools).

In [ ]:
import numpy as np
import anythreejs as p3

In [ ]:
# 1. The README scene: red box, standard material, two lights
scene = p3.Scene(
    children=[
        p3.Mesh(
            geometry=p3.BoxGeometry(1, 1, 1),
            material=p3.MeshStandardMaterial(color="#ff0c0c", roughness=0.4),
        ),
        p3.AmbientLight(intensity=0.4),
        p3.DirectionalLight(position=[5, 5, 5], intensity=1),
    ],
    background="#e5e5fa",
)
camera = p3.PerspectiveCamera(position=[3, 3, 3], aspect=700 / 450)
p3.Renderer(camera=camera, scene=scene,
            controls=[p3.OrbitControls(controlling=camera)],
            width=700, height=450)

In [ ]:
# 2. Torus with PBR shading — check specular highlight moves as you orbit
torus = p3.Mesh(
    geometry=p3.TorusGeometry(radius=1, tube=0.35),
    material=p3.MeshStandardMaterial(color="#3377cc", roughness=0.3, metalness=0.6),
)
scene = p3.Scene(
    children=[torus, p3.AmbientLight(intensity=0.3),
              p3.DirectionalLight(position=[5, 5, 5], intensity=1.5)],
    background="#101018",
)
camera = p3.PerspectiveCamera(position=[0, 0, 3.5], aspect=600 / 400)
p3.Renderer(camera=camera, scene=scene,
            controls=[p3.OrbitControls(controlling=camera)],
            width=600, height=400)

In [ ]:
# 3. Vertex-colored point cloud (plopp scatter3d's rendering path)
n = 50_000
rng = np.random.default_rng(0)
positions = (rng.random((n, 3), dtype="float32") - 0.5) * 4
colors = np.ones((n, 4), dtype="float32")
colors[:, :3] = positions / 4 + 0.5
geometry = p3.BufferGeometry(attributes={
    "position": p3.BufferAttribute(positions, itemSize=3),
    "color": p3.BufferAttribute(colors, itemSize=4),
})
cloud = p3.Points(geometry=geometry,
                  material=p3.PointsMaterial(size=2, vertexColors=True))
scene = p3.Scene(children=[cloud, p3.AxesHelper(size=2)], background="#181818")
camera = p3.PerspectiveCamera(position=[3, 3, 3], aspect=600 / 400)
p3.Renderer(camera=camera, scene=scene,
            controls=[p3.OrbitControls(controlling=camera)],
            width=600, height=400)

In [ ]:
# 4. Live updates: run this cell, then re-run it — colors must change
#    in the figure above WITHOUT re-rendering the scene (delta protocol)
colors[:, :3] = rng.random((n, 3)).astype("float32")
geometry.attributes["color"].array = colors

In [ ]:
# 5. Fat lines (Line2) — width in pixels, must survive zoom
theta = np.linspace(0, 6 * np.pi, 300, dtype="float32")
spiral = np.stack([np.cos(theta), np.sin(theta), theta / 10 - 1], axis=1)
line = p3.Line2(
    geometry=p3.LineGeometry(positions=spiral),
    material=p3.LineMaterial(color="#22cc66", linewidth=5),
)
scene = p3.Scene(children=[line], background="#ffffff")
camera = p3.PerspectiveCamera(position=[0, 0, 4], aspect=600 / 400)
p3.Renderer(camera=camera, scene=scene,
            controls=[p3.OrbitControls(controlling=camera)],
            width=600, height=400)

In [ ]:
# 6. Text sprites (plopp's axis labels) — text should stay camera-facing
sprites = [
    p3.Sprite(
        material=p3.SpriteMaterial(
            map=p3.TextTexture(string=label, color=color, size=64),
            transparent=True,
        ),
        position=pos, scale=[1.2, 0.4, 1.0],
    )
    for label, color, pos in [
        ("x-axis", "#cc2222", [2, 0, 0]),
        ("y-axis", "#22aa22", [0, 2, 0]),
        ("z-axis", "#2222cc", [0, 0, 2]),
    ]
]
scene = p3.Scene(children=[p3.AxesHelper(size=2), *sprites], background="#f8f8f0")
camera = p3.PerspectiveCamera(position=[3, 3, 3], aspect=600 / 400)
p3.Renderer(camera=camera, scene=scene,
            controls=[p3.OrbitControls(controlling=camera)],
            width=600, height=400)

In [ ]:
# 7. DataTexture on a plane (matplotgl's imshow path) — smooth gradient,
#    no banding, correct orientation (bright corner top-right)
h, w = 64, 64
img = np.zeros((h, w, 4), dtype="uint8")
img[..., 0] = np.linspace(0, 255, w, dtype="uint8")[None, :]
img[..., 1] = np.linspace(255, 0, h, dtype="uint8")[:, None]
img[..., 3] = 255
plane = p3.Mesh(
    geometry=p3.PlaneGeometry(3, 3),
    material=p3.MeshBasicMaterial(map=p3.DataTexture(data=img)),
)
scene = p3.Scene(children=[plane], background="#222222")
camera = p3.PerspectiveCamera(position=[0, 0, 3], aspect=600 / 400)
p3.Renderer(camera=camera, scene=scene,
            controls=[p3.OrbitControls(controlling=camera)],
            width=600, height=400)

In [ ]:
# 8. Full plopp figure (needs: uv pip install -e ./external/plopp scipp)
try:
    import plopp as pp
    from plopp.data.testing import scatter
    fig = pp.scatter3d(scatter(), x="x", y="y", z="z")
    display(fig)
except ImportError as error:
    print(f"plopp/scipp not installed - skipping ({error})")